In [3]:
import pandas as pd
import os
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

def get_metrics(y_true, y_pred):
    # Confusion Matrix: [[TN, FP], [FN, TP]]
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall_pos = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    recall_neg = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    return (tn, fp, fn, tp), precision, recall_pos, recall_neg, f1

def run_thermompnn_evaluation():
    # Loop through case1 to case6
    cases = [f"case{i}" for i in range(1, 7)]
    test_sets = [1, 2, 3]

    for case in cases:
        if not os.path.isdir(case):
            continue
            
        # Path where CSVs are: caseX/thermompnn_d_results/
        result_dir = os.path.join(case, "thermompnn_d_results")
        # Path where logs go: caseX/logs/
        log_dir = os.path.join(case, "logs")
        
        if not os.path.exists(result_dir):
            print(f"Skipping {case}: 'thermompnn_d_results' folder not found.")
            continue

        os.makedirs(log_dir, exist_ok=True)
        
        output_lines = [
            "Stability Prediction Evaluation (ThermoMPNN-D Model)",
            "Positive Class: Stable (experimental_ddG > 0, predicted_ddG > 0)",
            "Negative Class: Unstable (experimental_ddG <= 0, predicted_ddG <= 0)\n"
        ]

        for ts in test_sets:
            file_name = f"Test_Set_{ts}_thermompnn_d_scores.csv"
            file_path = os.path.join(result_dir, file_name)
            
            if not os.path.exists(file_path):
                continue
            
            # Load data
            df = pd.read_csv(file_path)
            
            # Data Cleaning: ThermoMPNN skips Indels (NaN). 
            # We must drop rows where either score is NaN to get a valid confusion matrix.
            df_clean = df.dropna(subset=['predicted_ddG', 'experimental_ddG'])
            
            if df_clean.empty:
                print(f"Warning: No valid rows in {file_path}")
                continue

            # --- logic ---
            y_true = (df_clean['experimental_ddG'] <= 0).astype(int)
            y_pred = (df_clean['predicted_ddG'] > 0).astype(int)
            
            metrics = get_metrics(y_true, y_pred)
            (tn, fp, fn, tp), prec, rec_p, rec_n, f1 = metrics
            
            output_lines.append(f"=== Test_Set_{ts} ===")
            output_lines.append(f"Total Pairs Evaluated: {len(df_clean)}")
            output_lines.append(f"Confusion Matrix (TN, FP, FN, TP):")
            output_lines.append(f"[[{tn} {fp}]\n [{fn} {tp}]]")
            output_lines.append(f"Precision: {prec:.4f}")
            output_lines.append(f"Recall (Stable/Pos): {rec_p:.4f}")
            output_lines.append(f"Recall (Unstable/Neg): {rec_n:.4f}")
            output_lines.append(f"F1 Score: {f1:.4f}")
            output_lines.append("-" * 50 + "\n")

        # Write the file to caseX/logs/
        output_file_path = os.path.join(log_dir, "final_stability_metrics.txt")
        with open(output_file_path, "w") as f:
            f.write("\n".join(output_lines))
        
        print(f"Metrics generated for {case} at {output_file_path}")

if __name__ == "__main__":
    run_thermompnn_evaluation()

Metrics generated for case1 at case1/logs/final_stability_metrics.txt
Metrics generated for case2 at case2/logs/final_stability_metrics.txt
Metrics generated for case3 at case3/logs/final_stability_metrics.txt
Metrics generated for case4 at case4/logs/final_stability_metrics.txt
Metrics generated for case5 at case5/logs/final_stability_metrics.txt
Metrics generated for case6 at case6/logs/final_stability_metrics.txt
